# Day 09 — HTTP Requests and APIs
**Vidya Sankalp | Python for AI Engineering**

## What you'll learn
- Understand client, server, request, and response
- Send HTTP requests with `httpx`
- Use GET, POST, PUT, PATCH, and DELETE
- Work with paths, query parameters, headers, and JSON
- Check status codes and handle errors
- Connect API responses to Pydantic models

We'll use a demo API and familiar **products, customers, and orders**. Some write operations on demo APIs are simulated; they do not permanently change a real store.

> **Setup:** Run `pip install httpx pydantic` if needed. The API examples require an internet connection.

## 1. What is an API?

Think of an API as a way for one program to ask another program for data or to request an action.

- **Client:** our Python program.
- **Server:** the remote program that handles the request.
- **Request:** what we send (method, URL, optional headers and body).
- **Response:** what comes back (status code, headers and body).
- **JSON:** a common format for exchanging structured data.

Our Python code is the client. We'll use DummyJSON as a practice server.

In [ ]:
import httpx

BASE_URL = "https://dummyjson.com"
print("HTTP client is ready.")

## 2. First request: GET a product

`GET` asks the server to return data. `httpx.get()` sends the request; `.status_code` tells us the result; `.json()` converts a JSON response into Python data.

In [ ]:
response = httpx.get(f"{BASE_URL}/products/1", timeout=10.0)

print("Status:", response.status_code)
print("Content type:", response.headers.get("content-type"))

if response.status_code == 200:
    product = response.json()
    print("Product:", product["title"])
    print("Price:", product["price"])

### Understand the URL

In `https://dummyjson.com/products/1`:
- `https://` is the protocol.
- `dummyjson.com` is the server.
- `/products/1` is the path: product with ID 1.

A response with status `200` usually means the request succeeded.

## 3. Read a list and use query parameters

A path identifies the resource. Query parameters let us filter or control the results. Use `params=` instead of manually building a query string.

In [ ]:
response = httpx.get(
    f"{BASE_URL}/products",
    params={"limit": 5},
    timeout=10.0
)

response.raise_for_status()
data = response.json()

print("Products returned:", len(data["products"]))
for item in data["products"]:
    print(item["id"], item["title"], "-", item["price"])

For example, `params={"limit": 5}` becomes `?limit=5` in the URL. `raise_for_status()` raises an exception if the server returns an unsuccessful HTTP status, such as 404 or 500.

In [ ]:
# Search products by a phrase.
response = httpx.get(
    f"{BASE_URL}/products/search",
    params={"q": "phone"},
    timeout=10.0
)
response.raise_for_status()

for item in response.json()["products"]:
    print(item["title"])

## 4. Common status codes

| Code | Meaning | Typical interpretation |
|---|---|---|
| `200` | OK | Request succeeded |
| `201` | Created | A resource was created |
| `400` | Bad Request | The request data is invalid |
| `401` | Unauthorized | Authentication is missing or invalid |
| `403` | Forbidden | You are not allowed to do this |
| `404` | Not Found | The resource does not exist |
| `429` | Too Many Requests | Rate limit reached |
| `500` | Internal Server Error | The server failed |

Don't assume every successful request returns exactly `200`. For example, creation may return `201`.

In [ ]:
# Try requesting a product ID that probably does not exist.
response = httpx.get(f"{BASE_URL}/products/999999", timeout=10.0)

print("Status:", response.status_code)
if response.status_code == 200:
    print(response.json())
elif response.status_code == 404:
    print("Product was not found.")
else:
    print("Request returned another status:", response.status_code)

## 5. Send data with POST

`POST` commonly sends data to create a resource. Put the data in a Python dictionary and pass it using `json=`.

In [ ]:
new_product = {
    "title": "Vidya Sankalp Notebook",
    "price": 12.50,
    "category": "stationery"
}

response = httpx.post(
    f"{BASE_URL}/products/add",
    json=new_product,
    timeout=10.0
)

print("Status:", response.status_code)
print("Response:", response.json())

DummyJSON simulates adding the product and returns a response, but the record is not permanently stored. In a real API, follow the service's documentation to understand its behavior and permissions.

**Remember:** `json=payload` asks `httpx` to encode the dictionary as JSON and set the appropriate content type.

## 6. Update with PUT or PATCH; remove with DELETE

- `PUT` commonly replaces a resource representation.
- `PATCH` commonly changes selected fields.
- `DELETE` requests removal of a resource.

Exact behavior depends on the API's documentation.

In [ ]:
# Demo update (simulated by DummyJSON).
response = httpx.patch(
    f"{BASE_URL}/products/1",
    json={"price": 99.99},
    timeout=10.0
)
print("PATCH status:", response.status_code)
print(response.json())

In [ ]:
# Demo delete (simulated by DummyJSON).
response = httpx.delete(f"{BASE_URL}/products/1", timeout=10.0)
print("DELETE status:", response.status_code)
print(response.json())

## 7. Headers and authentication

Headers carry extra information with a request. For example, `Accept` tells the server what response format the client prefers. Many APIs also require an API key or bearer token in an `Authorization` header.

Never hard-code real secrets in a notebook or commit them to source control.

In [ ]:
response = httpx.get(
    f"{BASE_URL}/products/1",
    headers={"Accept": "application/json"},
    timeout=10.0
)
print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))

## 8. Handle network and HTTP errors

Network calls can fail because of timeouts, connection problems, or unsuccessful HTTP responses. Catch errors you can reasonably handle instead of assuming every call succeeds.

In [ ]:
try:
    response = httpx.get(f"{BASE_URL}/products/1", timeout=5.0)
    response.raise_for_status()
    product = response.json()
    print("Product:", product["title"])

except httpx.TimeoutException:
    print("The request took too long. Try again later.")

except httpx.HTTPStatusError as error:
    print("The server returned an error status:", error.response.status_code)

except httpx.RequestError as error:
    print("Could not complete the request:", error)

A **timeout** is a limit on how long the client waits. It prevents your program from waiting indefinitely. A timeout does not necessarily mean the server did not process the request; be careful about automatically retrying operations that create or charge for something.

## 9. Connect an API response to Pydantic

HTTP gives us data; Pydantic can validate that data before our application uses it. This joins the concepts from Day 08.

In [ ]:
from pydantic import BaseModel, Field

class ProductSummary(BaseModel):
    id: int = Field(gt=0)
    title: str
    price: float = Field(ge=0)

response = httpx.get(f"{BASE_URL}/products/1", timeout=10.0)
response.raise_for_status()

raw_product = response.json()
product = ProductSummary.model_validate(raw_product)

print("ID:", product.id)
print("Title:", product.title)
print("Price:", product.price)

The response can contain many additional fields. This small model validates the fields our application needs.

**Pattern:** `HTTP request → response JSON → Pydantic validation → application logic`

## 10. Mini exercise: customer lookup

Use the demo users endpoint to retrieve one user and display their name and email. Then add basic error handling.

Hints:
- URL: `f"{BASE_URL}/users/1"`
- Send the request with `httpx.get(...)`
- Call `raise_for_status()`
- Read the JSON with `.json()`

In [ ]:
# Your turn: complete this cell.
# 1. Send a GET request to /users/1.
# 2. Check the response status.
# 3. Print the user's full name and email.

# response = ...
# response.raise_for_status()
# user = response.json()
# print(...)

## 11. Practice tasks

1. Fetch five products and print their titles.
2. Search products using a query parameter.
3. Request a product by ID using a path parameter.
4. Send a simulated POST request with a new product dictionary.
5. Use PATCH to change one field and print the response.
6. Add `try`/`except` handling for timeout and HTTP status errors.
7. Create a Pydantic model for a user with `id`, `firstName`, `lastName`, and `email`. Validate a response from `/users/1`.
8. Explain the difference between a path parameter and a query parameter.

## Day 09 recap

| Concept | Example |
|---|---|
| GET | `httpx.get(url)` |
| POST with JSON | `httpx.post(url, json=data)` |
| PUT / PATCH | Update a resource |
| DELETE | Request resource removal |
| Query parameters | `params={"limit": 5}` |
| Status code | `response.status_code` |
| JSON response | `response.json()` |
| Raise on HTTP error | `response.raise_for_status()` |
| Timeout | `timeout=5.0` |
| Error handling | Catch `httpx` exceptions |
| Validate API data | `Model.model_validate(response.json())` |

**Big idea:** an API call is a request to another program. Your code sends the request, checks the response, and validates the data before relying on it.